In [24]:
# ==========================================================
# 1. IMPORT LIBRARIES
# ==========================================================

import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout,
    Input
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
import gc

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import time

start_time = time.time()

###########################################################

# ==========================================================
# 2. LOAD MODELLING DATASETS
# ==========================================================

solar_model_df = pd.read_csv(
    "Data_Corpus/final/modelling/solar_model_dataset.csv"
)

wind_onshore_model_df = pd.read_csv(
    "Data_Corpus/final/modelling/wind_onshore_model_dataset.csv"
)

wind_offshore_model_df = pd.read_csv(
    "Data_Corpus/final/modelling/wind_offshore_model_dataset.csv"
)

for df in [
    solar_model_df,
    wind_onshore_model_df,
    wind_offshore_model_df
]:
    df["datetime"] = pd.to_datetime(df["datetime"])

    
DATASETS = {
    "Solar": solar_model_df,
    "Wind_Onshore": wind_onshore_model_df,
    "Wind_Offshore": wind_offshore_model_df
}
######################################################33

PERIODS = {

    "FULL": {
        "start": "2015-01-08 00:00:00",
        "train_end": "2022-12-31 23:00:00",
        "valid_end": "2023-12-31 23:00:00"
    },

    "POST_2020": {
        "start": "2020-01-01 00:00:00",
        "train_end": "2023-12-31 23:00:00",
        "valid_end": "2024-12-31 23:00:00"
    },

    "POST_2021": {
        "start": "2021-01-01 00:00:00",
        "train_end": "2023-12-31 23:00:00",
        "valid_end": "2024-12-31 23:00:00"
    }
}

############################################################

HORIZONS = [    1,    6,    24,    72,    168]

###################################

# ==========================================================
# EPOCH SETTINGS
# ==========================================================

EPOCHS_LIST = [    20,    40,    60,    80,    100 ]

########################################3


results = pd.DataFrame(
    columns=[
        "Period",
        "Target",
        "Horizon",
        "Epochs",
        "Model",
        "MAE",
        "RMSE",
        "MAPE",
        "R2"
    ]
)

###################################################
# ==========================================================
# MAPE
# ==========================================================

def calculate_mape(
    y_true,
    y_pred
):

    y_true = np.where(
        y_true == 0,
        1e-8,
        y_true
    )

    mape = np.mean(
        np.abs(
            (y_true - y_pred)
            / y_true
        )
    ) * 100

    return mape


###############################################
def evaluate_model(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mape = calculate_mape(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return (
        mae,
        rmse,
        mape,
        r2
    )
##############################################

def get_period_split(df, period_name):

    period = PERIODS[period_name]

    df_period = df[
        df["datetime"] >= period["start"]
    ].copy()

    train_df = df_period[
        df_period["datetime"] <= period["train_end"]
    ]

    valid_df = df_period[
        (df_period["datetime"] > period["train_end"]) &
        (df_period["datetime"] <= period["valid_end"])
    ]

    test_df = df_period[
        df_period["datetime"] > period["valid_end"]
    ]

    return train_df, valid_df, test_df

##########################################################

LOOKBACK = 24

def create_sequences(
    X,
    y,
    lookback=24
):

    X_seq = []
    y_seq = []

    for i in range(
        lookback,
        len(X)
    ):

        X_seq.append(
            X[i-lookback:i]
        )

        y_seq.append(
            y[i]
        )

    return (
        np.array(X_seq),
        np.array(y_seq)
    )

    

#################################

def add_result(
    results,
    period,
    target,
    horizon,
    epochs,
    model,
    mae,
    rmse,
    mape,
    r2
):

    results.loc[len(results)] = {

        "Period": period,
        "Target": target,
        "Horizon": horizon,
        "Epochs": epochs,
        "Model": model,

        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    }

    return results




# ==========================================================
# 10. LSTM MODEL FUNCTION
# ==========================================================

def run_lstm(
    X_train,
    y_train,
    X_valid,
    y_valid,
    epochs=20,
    lookback=24
):

    # ==========================================
    # Scale Features
    # ==========================================
    feature_scaler = MinMaxScaler()

    X_train_scaled = feature_scaler.fit_transform(
        X_train
    )

    X_valid_scaled = feature_scaler.transform(
        X_valid
    )

    # ==========================================
    # Scale Target
    # ==========================================
    target_scaler = MinMaxScaler()

    y_train_scaled = target_scaler.fit_transform(
        y_train.values.reshape(-1,1)
    )

    y_valid_scaled = target_scaler.transform(
        y_valid.values.reshape(-1,1)
    )

    # ==========================================
    # Create Sequences
    # ==========================================
    X_train_seq, y_train_seq = create_sequences(
        X_train_scaled,
        y_train_scaled,
        lookback
    )

    X_valid_seq, y_valid_seq = create_sequences(
        X_valid_scaled,
        y_valid_scaled,
        lookback
    )

    # ==========================================
    # Build Model
    # ==========================================
    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train_seq.shape[1],
                X_train_seq.shape[2]
            )
        )
    )

    model.add(
        LSTM(
            64
        )
    )

    model.add(
        Dropout(0.2)
    )

    model.add(
        Dense(1)
    )

    model.compile(
        optimizer="adam",
        loss="mse"
    )

    # ==========================================
    # Early Stopping
    # ==========================================
    # early_stop = EarlyStopping(
    #     monitor="val_loss",
    #     patience=3,
    #     restore_best_weights=True
    # )

    # ==========================================
    # Train
    # ==========================================
    model.fit(
        X_train_seq,
        y_train_seq,
        validation_data=(
            X_valid_seq,
            y_valid_seq
        ),
        epochs=epochs,
        batch_size=256,
        # callbacks=[early_stop],
        verbose=0
    )

    # ==========================================
    # Predict
    # ==========================================
    y_pred_scaled = model.predict(
        X_valid_seq,
        verbose=0
    )

    # ==========================================
    # Inverse Scaling
    # ==========================================
    y_pred = target_scaler.inverse_transform(
        y_pred_scaled
    )

    y_true = target_scaler.inverse_transform(
        y_valid_seq
    )

    # ==========================================
    # Metrics
    # ==========================================
    mae, rmse, mape, r2 = evaluate_model(
        y_true,
        y_pred
    )
    # K.clear_session()
    # gc.collect()
    return model, mae, rmse,mape, r2

####################################################################3

# ==========================================================
# 4. FEATURE DEFINITIONS
# ==========================================================

SOLAR_FEATURES = [

    "global_radiation",
    "sunshine_duration",
    "solar_zenith_angle",
    "air_temperature",
    "relative_humidity",

    "capacity_photovoltaics_mw",

    "grid_load_mwh",
    "price_eur_per_mwh",

    "hour",
    "month",
    "dayofweek",

    "generation_photovoltaics_mwh_lag_1",
    "generation_photovoltaics_mwh_lag_6",
    "generation_photovoltaics_mwh_lag_24",
    "generation_photovoltaics_mwh_lag_72",
    "generation_photovoltaics_mwh_lag_168",

    "generation_photovoltaics_mwh_roll_mean_24",
    "generation_photovoltaics_mwh_roll_mean_168",
    "generation_photovoltaics_mwh_roll_std_24"
]
####################################
WIND_ON_FEATURES = [

    "wind_speed",
    "wind_dir_sin",
    "wind_dir_cos",

    "air_temperature",
    "relative_humidity",

    "capacity_wind_onshore_mw",

    "grid_load_mwh",
    "price_eur_per_mwh",

    "hour",
    "month",
    "dayofweek",

    "generation_wind_onshore_mwh_lag_1",
    "generation_wind_onshore_mwh_lag_6",
    "generation_wind_onshore_mwh_lag_24",
    "generation_wind_onshore_mwh_lag_72",
    "generation_wind_onshore_mwh_lag_168",

    "generation_wind_onshore_mwh_roll_mean_24",
    "generation_wind_onshore_mwh_roll_mean_168",
    "generation_wind_onshore_mwh_roll_std_24"
]
##################################

WIND_OFF_FEATURES = [

    "wind_speed",
    "wind_dir_sin",
    "wind_dir_cos",

    "air_temperature",
    "relative_humidity",

    "capacity_wind_offshore_mw",

    "grid_load_mwh",
    "price_eur_per_mwh",

    "hour",
    "month",
    "dayofweek",

    "generation_wind_offshore_mwh_lag_1",
    "generation_wind_offshore_mwh_lag_6",
    "generation_wind_offshore_mwh_lag_24",
    "generation_wind_offshore_mwh_lag_72",
    "generation_wind_offshore_mwh_lag_168",

    "generation_wind_offshore_mwh_roll_mean_24",
    "generation_wind_offshore_mwh_roll_mean_168",
    "generation_wind_offshore_mwh_roll_std_24"
]

##############################################
FEATURES = {
    "Solar": SOLAR_FEATURES,
    "Wind_Onshore": WIND_ON_FEATURES,
    "Wind_Offshore": WIND_OFF_FEATURES
}










def get_model_data(
    train_df,
    valid_df,
    test_df,
    target_name,
    horizon
):

    features = FEATURES[target_name]

    target_col = f"target_h{horizon}"

    X_train = train_df[features]
    y_train = train_df[target_col]

    X_valid = valid_df[features]
    y_valid = valid_df[target_col]

    X_test = test_df[features]
    y_test = test_df[target_col]

    return (
        X_train,
        y_train,
        X_valid,
        y_valid,
        X_test,
        y_test
    )









In [25]:
start_time = time.time()

TOTAL_RUNS = (
    len(PERIODS)
    * len(DATASETS)
    * len(HORIZONS)
    * len(EPOCHS_LIST)
)

counter = 1

print(f"Total Experiments = {TOTAL_RUNS}")

for period_name in PERIODS:

    for target_name in DATASETS:

        dataset = DATASETS[target_name]

        train_df, valid_df, test_df = get_period_split(
            dataset,
            period_name
        )

        for horizon in HORIZONS:

            (
                X_train,
                y_train,
                X_valid,
                y_valid,
                X_test,
                y_test
            ) = get_model_data(
                train_df,
                valid_df,
                test_df,
                target_name,
                horizon
            )

            for epochs in EPOCHS_LIST:

                print(
                    f"\n[{counter}/{TOTAL_RUNS}] "
                    f"{period_name} | "
                    f"{target_name} | "
                    f"H{horizon} | "
                    f"Epochs={epochs}"
                )

                model, mae, rmse,mape, r2 = run_lstm(
                    X_train,
                    y_train,
                    X_valid,
                    y_valid,
                    epochs=epochs
                )

                results = add_result(
                    results,
                    period_name,
                    target_name,
                    horizon,
                    epochs,
                    "LSTM",
                    mae,
                    rmse,
                    mape,
                    r2
                )

                results.to_csv(
                    "results/lstm_results.csv",
                    index=False
                )

                K.clear_session()
                gc.collect()

                counter += 1


################################################
end_time = time.time()

print(
    f"Total runtime: "
    f"{(end_time-start_time)/60:.2f} minutes"
)



Total Experiments = 225

[1/225] FULL | Solar | H1 | Epochs=20

[2/225] FULL | Solar | H1 | Epochs=40

[3/225] FULL | Solar | H1 | Epochs=60

[4/225] FULL | Solar | H1 | Epochs=80

[5/225] FULL | Solar | H1 | Epochs=100

[6/225] FULL | Solar | H6 | Epochs=20

[7/225] FULL | Solar | H6 | Epochs=40

[8/225] FULL | Solar | H6 | Epochs=60

[9/225] FULL | Solar | H6 | Epochs=80

[10/225] FULL | Solar | H6 | Epochs=100

[11/225] FULL | Solar | H24 | Epochs=20

[12/225] FULL | Solar | H24 | Epochs=40

[13/225] FULL | Solar | H24 | Epochs=60

[14/225] FULL | Solar | H24 | Epochs=80

[15/225] FULL | Solar | H24 | Epochs=100

[16/225] FULL | Solar | H72 | Epochs=20

[17/225] FULL | Solar | H72 | Epochs=40

[18/225] FULL | Solar | H72 | Epochs=60

[19/225] FULL | Solar | H72 | Epochs=80

[20/225] FULL | Solar | H72 | Epochs=100

[21/225] FULL | Solar | H168 | Epochs=20

[22/225] FULL | Solar | H168 | Epochs=40

[23/225] FULL | Solar | H168 | Epochs=60

[24/225] FULL | Solar | H168 | Epochs=80

[2

In [29]:
lstm_results_df = pd.read_csv("results/lstm_results.csv")
lstm_results_df

,Period,Target,Horizon,Epochs,Model,MAE,RMSE,MAPE,R2
0,FULL,Solar,1,20,LSTM,602.603506,1020.352217,1.461434e+03,0.989252
1,FULL,Solar,1,40,LSTM,442.585734,797.031427,1.025014e+03,0.993442
2,FULL,Solar,1,60,LSTM,433.569274,780.562896,6.515794e+02,0.993710
3,FULL,Solar,1,80,LSTM,422.977549,758.602883,6.653812e+02,0.994059
4,FULL,Solar,1,100,LSTM,416.559941,758.939807,9.254263e+02,0.994054
...,...,...,...,...,...,...,...,...,...
220,POST_2021,Wind_Offshore,168,20,LSTM,1657.037239,1978.620951,3.172745e+10,-0.101546
221,POST_2021,Wind_Offshore,168,40,LSTM,1923.321616,2397.456096,2.592488e+10,-0.617255
222,POST_2021,Wind_Offshore,168,60,LSTM,1846.683348,2301.167081,2.646914e+10,-0.489957
223,POST_2021,Wind_Offshore,168,80,LSTM,1862.503082,2318.059423,3.779020e+10,-0.511912


In [32]:
# ==========================================================
# BEST EPOCH PER PERIOD + TARGET + HORIZON
# ==========================================================

best_epochs = lstm_results_df.loc[
    lstm_results_df.groupby(
        ["Period", "Target", "Horizon"]
    )["RMSE"].idxmin()
].reset_index(drop=True)

best_epochs

,Period,Target,Horizon,Epochs,Model,MAE,RMSE,MAPE,R2
0,FULL,Solar,1,80,LSTM,422.977549,758.602883,6.653812e+02,0.994059
1,FULL,Solar,6,40,LSTM,1176.750211,2156.634335,2.678298e+03,0.951984
2,FULL,Solar,24,20,LSTM,1505.825810,2841.549972,2.001145e+03,0.916642
3,FULL,Solar,72,20,LSTM,2232.319667,3695.451194,1.099257e+04,0.859017
4,FULL,Solar,168,20,LSTM,2165.183462,4092.128412,3.390104e+03,0.827150
5,FULL,Wind_Offshore,1,60,LSTM,534.698492,737.371844,2.124459e+08,0.837335
6,FULL,Wind_Offshore,6,20,LSTM,983.611508,1276.898913,6.824792e+08,0.512382
7,FULL,Wind_Offshore,24,20,LSTM,1498.543132,1865.873582,4.292729e+09,-0.043839
8,FULL,Wind_Offshore,72,20,LSTM,1668.188934,2029.568156,5.431591e+09,-0.236828
9,FULL,Wind_Offshore,168,20,LSTM,1649.510638,2030.450243,7.548251e+09,-0.250306
